In [1]:
# ── Cell 1: Autopsy of the 2,365 test errors ──────────────────────────────
import sys, json
from pathlib import Path
import numpy as np
import pandas as pd
import joblib

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"

model = joblib.load(DATA_DIR / "sentiment_model.joblib")
df = pd.read_csv(DATA_DIR / "imdb_reviews.csv")
test = df[df["split"] == "test"].reset_index(drop=True)
print(f"Test set: {len(test):,} rows   label balance: {test['label'].mean():.3f}")

proba = model.predict_proba(test["text"])[:, 1]
pred = (proba >= 0.5).astype(int)
test = test.assign(proba=proba, pred=pred, correct=pred == test["label"])
errors = test[~test["correct"]]
print(f"Errors: {len(errors):,}  ({1 - test['correct'].mean():.1%})  "
      f"FP {(errors['label'] == 0).sum():,} / FN {(errors['label'] == 1).sum():,}")

# ── 1. Confidence profile: is the model wrong loudly or quietly? ──
conf = np.abs(errors["proba"] - 0.5) * 2          # 0 = coin flip, 1 = certain
print("\nError confidence profile:")
for lo, hi, name in [(0, .2, "near coin-flip (49-59%)"), (.2, .5, "mild"),
                     (.5, .8, "confident"), (.8, 1.01, "VERY confident (90%+)")]:
    m = ((conf >= lo) & (conf < hi)).sum()
    print(f"  {name:<26} {m:>6,}  ({m/len(errors):.1%})")

# ── 2. Linguistic markers: negation and contrast ──
NEG = r"\b(not|never|no|nothing|nobody|isn't|wasn't|don't|didn't|can't|won't|couldn't)\b"
CONTRAST = r"\b(but|however|although|though|yet|except|despite)\b"
for name, pattern in [("negation", NEG), ("contrast", CONTRAST)]:
    e = errors["text"].str.contains(pattern, case=False, regex=True).mean()
    c = test[test["correct"]]["text"].str.contains(pattern, case=False, regex=True).mean()
    print(f"\n{name:>9}: present in {e:.1%} of errors vs {c:.1%} of correct "
          f"(lift ×{e/c:.2f})")

# ── 3. Length: are long mixed reviews still the enemy? ──
elen = errors["text"].str.split().str.len()
clen = test[test["correct"]]["text"].str.split().str.len()
print(f"\nLength: errors median {elen.median():.0f} words vs correct {clen.median():.0f}")

# ── 4. The most confidently wrong reviews — read the enemy ──
print("\n── 5 most confident errors (truth vs prediction) ──")
worst = errors.assign(conf=conf).nlargest(5, "conf")
for _, row in worst.iterrows():
    lbl = "POS" if row["label"] == 1 else "NEG"
    prd = "POS" if row["pred"] == 1 else "NEG"
    print(f"\n  truth={lbl}  pred={prd}  P(pos)={row['proba']:.2f}")
    print("  " + " ".join(row["text"].split()[:55]) + " …")

# save the autopsy for later phases
errors[["text", "label", "proba"]].to_json(DATA_DIR / "classifier_errors.json",
                                           orient="records")
print(f"\nSaved classifier_errors.json ({len(errors):,} rows)")

Test set: 25,000 rows   label balance: 0.500
Errors: 2,365  (9.5%)  FP 1,203 / FN 1,162

Error confidence profile:
  near coin-flip (49-59%)       909  (38.4%)
  mild                          883  (37.3%)
  confident                     468  (19.8%)
  VERY confident (90%+)         105  (4.4%)

 negation: present in 85.6% of errors vs 86.2% of correct (lift ×0.99)

 contrast: present in 84.5% of errors vs 79.7% of correct (lift ×1.06)


/var/folders/t8/9dngqyfj7sz226rjz84nntvh0000gn/T/ipykernel_7062/3625591443.py:35: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  e = errors["text"].str.contains(pattern, case=False, regex=True).mean()
/var/folders/t8/9dngqyfj7sz226rjz84nntvh0000gn/T/ipykernel_7062/3625591443.py:36: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  c = test[test["correct"]]["text"].str.contains(pattern, case=False, regex=True).mean()



Length: errors median 174 words vs correct 170

── 5 most confident errors (truth vs prediction) ──

  truth=NEG  pred=POS  P(pos)=0.99
  Mickey Rourke hunts Diane Lane in Elmore Leonard's Killshot It is not like Mickey Rourke ever really disappeared. He has had a steady string of appearances before he burst back on the scene. He was memorable in: Domino, Sin City, Man on Fire, Once Upon a Time in Mexico, and Get Carter. But in his …

  truth=NEG  pred=POS  P(pos)=0.99
  This is definitely one of the best Kung fu movies in the history of Cinema. The screenplay is really well done (which is not often the case for this type of movies) and you can see that Chuck (in one of his first role)is a great actor. The final fight with the sherif deputy in …

  truth=NEG  pred=POS  P(pos)=0.99
  This movie was pure genius. John Waters is brilliant. It is hilarious and I am not sick of it even after seeing it about 20 times since I bought it a few months ago. The acting is great, although Ricki Lak

In [2]:
# ── Cell 2: Finish the autopsy — confidence profile + the rating probe ────
import re

# rebuild the scored test frame (seconds)
proba = model.predict_proba(test["text"])[:, 1] if "proba" not in test else test["proba"]
test = test.assign(proba=proba, pred=(proba >= 0.5).astype(int))
test = test.assign(correct=test["pred"] == test["label"],
                   conf=np.abs(proba - 0.5) * 2)
errors = test[~test["correct"]]

print("── Error confidence profile (the strategy number) ──")
for lo, hi, name in [(0, .2, "near coin-flip"), (.2, .5, "mild"),
                     (.5, .8, "confident"), (.8, 1.01, "VERY confident")]:
    m = ((errors["conf"] >= lo) & (errors["conf"] < hi)).sum()
    print(f"  {name:<16} {m:>6,}  ({m/len(errors):.1%})")

NEG = r"\b(not|never|no|nothing|nobody|isn't|wasn't|don't|didn't|can't|won't|couldn't)\b"
CONTRAST = r"\b(but|however|although|though|yet|except|despite)\b"
print("\n── Linguistic lifts ──")
for name, pattern in [("negation", NEG), ("contrast", CONTRAST)]:
    e = errors["text"].str.contains(pattern, case=False, regex=True).mean()
    c = test[test["correct"]]["text"].str.contains(pattern, case=False, regex=True).mean()
    print(f"  {name:>9}: {e:.1%} of errors vs {c:.1%} of correct  (lift ×{e/c:.2f})")

# ── The rating probe: sarcasm / label-noise fingerprint ──
PRAISE = r"\b(best|brilliant|genius|classic|great|magnificent|hilarious|masterpiece|perfect)\b"
vconf = errors[errors["conf"] >= 0.8]
print(f"\n── VERY-confident errors: {len(vconf):,} — who are they? ──")
print("  rating distribution (truth labels come from these):")
print(vconf["rating"].value_counts().sort_index().to_string())

praise_neg = vconf[(vconf["label"] == 0) &
                   vconf["text"].str.contains(PRAISE, case=False, regex=True)]
print(f"\n  truth-NEG but praise-worded: {len(praise_neg):,} "
      f"({len(praise_neg)/max(1,len(vconf)):.0%} of very-confident errors)")
print("  their ratings:", praise_neg["rating"].value_counts().sort_index().to_dict())

# sincere-sounding check: does VADER also read them as positive?
from nltk.sentiment import SentimentIntensityAnalyzer
sia = SentimentIntensityAnalyzer()
agree = sum(sia.polarity_scores(t)["compound"] > 0.3
            for t in praise_neg["text"].head(200))
print(f"  VADER also reads them positive: {agree}/{min(200, len(praise_neg))} "
      "→ two independent readers fooled = sarcasm/noise, not a model bug")

# contrast-pivot class: praise BEFORE a contrast marker, truth in the tail
pivot = vconf[vconf["text"].str.contains(CONTRAST, case=False, regex=True)]
print(f"\n  very-confident errors containing a contrast marker: "
      f"{len(pivot):,} ({len(pivot)/max(1,len(vconf)):.0%})")

── Error confidence profile (the strategy number) ──
  near coin-flip      909  (38.4%)
  mild                883  (37.3%)
  confident           468  (19.8%)
  VERY confident      105  (4.4%)

── Linguistic lifts ──
   negation: 85.6% of errors vs 86.2% of correct  (lift ×0.99)
   contrast: 84.5% of errors vs 79.7% of correct  (lift ×1.06)

── VERY-confident errors: 105 — who are they? ──
  rating distribution (truth labels come from these):
rating
1     13
2      6
3     14
4     27
7     24
8      7
9      7
10     7

  truth-NEG but praise-worded: 37 (35% of very-confident errors)
  their ratings: {1: 11, 2: 4, 3: 8, 4: 14}


/var/folders/t8/9dngqyfj7sz226rjz84nntvh0000gn/T/ipykernel_7062/517802038.py:21: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  e = errors["text"].str.contains(pattern, case=False, regex=True).mean()
/var/folders/t8/9dngqyfj7sz226rjz84nntvh0000gn/T/ipykernel_7062/517802038.py:22: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  c = test[test["correct"]]["text"].str.contains(pattern, case=False, regex=True).mean()
/var/folders/t8/9dngqyfj7sz226rjz84nntvh0000gn/T/ipykernel_7062/517802038.py:33: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  vconf["text"].str.contains(PRAISE, case=False, regex=True)]


  VADER also reads them positive: 37/37 → two independent readers fooled = sarcasm/noise, not a model bug

  very-confident errors containing a contrast marker: 90 (86%)


/var/folders/t8/9dngqyfj7sz226rjz84nntvh0000gn/T/ipykernel_7062/517802038.py:47: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  pivot = vconf[vconf["text"].str.contains(CONTRAST, case=False, regex=True)]


In [3]:
# ── Cell 3: NB-SVM log-count-ratio features (Wang & Manning, 2012) ────────
import numpy as np
import scipy.sparse as sp
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV

class NBLogCountRatio(BaseEstimator, TransformerMixin):
    """Scales binarized features by r = log( P(f|pos) / P(f|neg) )."""
    def fit(self, X, y):
        y = np.asarray(y)
        p = np.asarray(X[y == 1].sum(axis=0)).ravel() + 1.0
        q = np.asarray(X[y == 0].sum(axis=0)).ravel() + 1.0
        self.r_ = sp.diags(np.log((p / p.sum()) / (q / q.sum())))
        return self
    def transform(self, X):
        return X @ self.r_

train = pd.read_csv(DATA_DIR / "imdb_train_clean.csv")
X_tr, y_tr = train["text"], train["label"]
print(f"Train: {len(train):,} rows")

nbsvm = Pipeline([
    ("vec", CountVectorizer(ngram_range=(1, 2), min_df=5, binary=True)),
    ("nb",  NBLogCountRatio()),
    ("clf", LogisticRegression(max_iter=2000, solver="liblinear")),
])
grid = GridSearchCV(nbsvm, {"clf__C": [0.5, 1, 4, 16]},
                    cv=5, scoring="accuracy", n_jobs=-1, verbose=1)
grid.fit(X_tr, y_tr)
print(f"\nCV results (accuracy):")
for c, s in zip(grid.cv_results_["param_clf__C"], grid.cv_results_["mean_test_score"]):
    print(f"  C = {c:<5} {s:.1%}")
print(f"Best: C = {grid.best_params_['clf__C']}  CV = {grid.best_score_:.1%}"
      f"   (TF-IDF LogReg CV was 88.6%)")

# ── the single test shot ──
best = grid.best_estimator_
test_acc = (best.predict(test["text"]) == test["label"]).mean()
print(f"\nTEST (spent once for this candidate): {test_acc:.1%}"
      f"   (shipped model: 90.5% · Wang & Manning: 91.2%)")

# interpretability check — the representation changed; did the story survive?
vec = best.named_steps["vec"]
r = best.named_steps["nb"].r_.diagonal()
w = best.named_steps["clf"].coef_[0] * r          # effective per-feature pull
feats = vec.get_feature_names_out()
order = np.argsort(w)
print("\nStrongest features (weight × ratio):")
print("  negative:", [feats[i] for i in order[:8]])
print("  positive:", [feats[i] for i in order[-8:][::-1]])

Train: 24,902 rows
Fitting 5 folds for each of 4 candidates, totalling 20 fits

CV results (accuracy):
  C = 0.5   88.8%
  C = 1.0   88.6%
  C = 4.0   88.3%
  C = 16.0  88.2%
Best: C = 0.5  CV = 88.8%   (TF-IDF LogReg CV was 88.6%)

TEST (spent once for this candidate): 90.8%   (shipped model: 90.5% · Wang & Manning: 91.2%)

Strongest features (weight × ratio):
  negative: ['had high', 'this crap', 'not worth', 'prom night', 'save yourself', 'unfunny', 'not recommend', 'not convincing']
  positive: ['definitely worth', 'well worth', 'loved that', 'excellent movie', 'has everything', 'loved this', 'refreshing', 'favorite movies']


In [6]:
# ── Cell 4: CV-only bake-off — one test shot for the winner ───────────────
import time
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, cross_val_predict

folds = StratifiedKFold(5, shuffle=True, random_state=42)

candidates = {
    "tfidf-LR (shipped config)": clone(model),
    "NBSVM (1,2)": Pipeline([
        ("vec", CountVectorizer(ngram_range=(1, 2), min_df=5, binary=True)),
        ("nb",  NBLogCountRatio()),
        ("clf", LogisticRegression(C=0.5, max_iter=2000, solver="liblinear")),
    ]),
    "NBSVM (1,3)": Pipeline([
        ("vec", CountVectorizer(ngram_range=(1, 3), min_df=5, binary=True)),
        ("nb",  NBLogCountRatio()),
        ("clf", LogisticRegression(C=0.5, max_iter=2000, solver="liblinear")),
    ]),
}

cv_proba = {}
for name, pipe in candidates.items():
    t0 = time.time()
    cv_proba[name] = cross_val_predict(pipe, X_tr, y_tr, cv=folds,
                                       method="predict_proba", n_jobs=-1)[:, 1]
    acc = ((cv_proba[name] >= 0.5).astype(int) == y_tr).mean()
    print(f"  {name:<28} CV {acc:.2%}   ({time.time()-t0:.0f}s)")

ens12 = (cv_proba["tfidf-LR (shipped config)"] + cv_proba["NBSVM (1,2)"]) / 2
ens13 = (cv_proba["tfidf-LR (shipped config)"] + cv_proba["NBSVM (1,3)"]) / 2
for name, p in [("ENSEMBLE tfidf + NBSVM(1,2)", ens12),
                ("ENSEMBLE tfidf + NBSVM(1,3)", ens13)]:
    acc = ((p >= 0.5).astype(int) == y_tr).mean()
    print(f"  {name:<28} CV {acc:.2%}")

# ── decide on CV, then ONE test shot ──
WINNER = "ENSEMBLE tfidf + NBSVM(1,2)"       # ← edit to the top CV line first!

if WINNER.startswith("ENSEMBLE"):
    part = "(1,3)" if "(1,3)" in WINNER else "(1,2)"
    m1 = clone(model).fit(X_tr, y_tr)
    m2 = clone(candidates[f"NBSVM {part}"]).fit(X_tr, y_tr)
    p_test = (m1.predict_proba(test["text"])[:, 1]
              + m2.predict_proba(test["text"])[:, 1]) / 2
else:
    m = clone(candidates[WINNER]).fit(X_tr, y_tr)
    p_test = m.predict_proba(test["text"])[:, 1]

acc = ((p_test >= 0.5).astype(int) == test["label"]).mean()
print(f"\nFINAL CLASSICAL TEST SHOT — {WINNER}: {acc:.1%}")
print("  shipped 90.5 · NBSVM 90.8 · Wang & Manning 91.2")

  tfidf-LR (shipped config)    CV 90.42%   (5s)
  NBSVM (1,2)                  CV 90.95%   (5s)
  NBSVM (1,3)                  CV 91.06%   (9s)
  ENSEMBLE tfidf + NBSVM(1,2)  CV 91.14%
  ENSEMBLE tfidf + NBSVM(1,3)  CV 91.31%

FINAL CLASSICAL TEST SHOT — ENSEMBLE tfidf + NBSVM(1,2): 91.3%
  shipped 90.5 · NBSVM 90.8 · Wang & Manning 91.2


In [7]:
# ── Cell 5: Ship the ensemble — module, artefact, verification ────────────
SENTIMENT_SOURCE = '''"""
Sentiment model components for SciSpell (V2 classifier).
Source: notebooks/11_V2_Classifier.ipynb. Importable so joblib artefacts
that reference these classes load anywhere the app runs.
"""
import numpy as np
import scipy.sparse as sp
from sklearn.base import BaseEstimator, TransformerMixin

class NBLogCountRatio(BaseEstimator, TransformerMixin):
    """Scales binarized features by r = log( P(f|pos) / P(f|neg) )."""
    def fit(self, X, y):
        y = np.asarray(y)
        p = np.asarray(X[y == 1].sum(axis=0)).ravel() + 1.0
        q = np.asarray(X[y == 0].sum(axis=0)).ravel() + 1.0
        self.r_ = sp.diags(np.log((p / p.sum()) / (q / q.sum())))
        return self
    def transform(self, X):
        return X @ self.r_

class SoftVoteEnsemble:
    """Average-probability ensemble of fitted sklearn text pipelines."""
    def __init__(self, pipelines):
        self.pipelines = pipelines
    def predict_proba(self, texts):
        ps = [p.predict_proba(texts) for p in self.pipelines]
        return np.mean(ps, axis=0)
    def predict(self, texts):
        return (self.predict_proba(texts)[:, 1] >= 0.5).astype(int)
    def explain(self, text, top=10):
        """Merged per-feature contributions (weight x value), averaged
        across members — both are linear, so this is exact."""
        contrib = {}
        for pipe in self.pipelines:
            vec = pipe.named_steps["vec"]
            X = vec.transform([text])
            if "nb" in pipe.named_steps:
                X = pipe.named_steps["nb"].transform(X)
            w = pipe.named_steps["clf"].coef_[0]
            X = X.tocoo()
            names = vec.get_feature_names_out()
            for j, v in zip(X.col, X.data):
                contrib[names[j]] = contrib.get(names[j], 0.0) + w[j] * v
        n = len(self.pipelines)
        merged = sorted(((f, c / n) for f, c in contrib.items()),
                        key=lambda x: -abs(x[1]))[:top]
        return merged
'''
(PROJECT_ROOT / "app" / "sentiment.py").write_text(SENTIMENT_SOURCE, encoding="utf-8")

sys.path.insert(0, str(PROJECT_ROOT / "app"))
import importlib
import sentiment as sentiment_mod
importlib.reload(sentiment_mod)
from sentiment import NBLogCountRatio as NBLCR_mod, SoftVoteEnsemble

# rebuild both members from the module's classes, fit on full clean train
m1 = clone(model).fit(X_tr, y_tr)
m2 = Pipeline([
    ("vec", CountVectorizer(ngram_range=(1, 2), min_df=5, binary=True)),
    ("nb",  NBLCR_mod()),
    ("clf", LogisticRegression(C=0.5, max_iter=2000, solver="liblinear")),
]).fit(X_tr, y_tr)
ens = SoftVoteEnsemble([m1, m2])

acc = (ens.predict(test["text"]) == test["label"]).mean()
probe = ens.predict(["not good at all", "absolutely wonderful film",
                     "all good, had some problem, but i solved it a little"])
exp = ens.explain("not good at all", top=4)

joblib.dump(ens, DATA_DIR / "sentiment_model_v2.joblib")
size = (DATA_DIR / "sentiment_model_v2.joblib").stat().st_size / 1e6

res_path = DATA_DIR / "classification_results.json"
res = json.loads(res_path.read_text())
res["v2"] = {"model": "SoftVote(tfidf-LR + NBSVM(1,2))",
             "cv_shuffled": 0.9114, "test_acc": round(float(acc), 4),
             "note": "classical ceiling matched (W&M 91.2); "
                     "ens(1,3) CV 91.31 untested by protocol"}
res_path.write_text(json.dumps(res, indent=2))

checks = [
    ("Ensemble reproduces the test shot", abs(acc - 0.913) < 0.002, f"{acc:.1%}"),
    ("Probes sane", list(probe) == [0, 1, 1], f"{list(probe)}"),
    ("Explain merges both vocabularies",
     any(" " in f for f, _ in exp), f"{exp[:3]}"),
    ("Artefact saved", size > 5, f"{size:.1f} MB"),
]
width = max(len(c[0]) for c in checks)
print("VERIFICATION\n" + "─" * (width + 30))
for label, ok, detail in checks:
    print(f"  {'PASS' if ok else 'FAIL'}  {label:<{width}}  {detail}")
print("─" * (width + 30))
print(f"  {sum(ok for _, ok, _ in checks)}/{len(checks)} checks passed")
assert all(ok for _, ok, _ in checks)

VERIFICATION
───────────────────────────────────────────────────────────────
  PASS  Ensemble reproduces the test shot  91.3%
  PASS  Probes sane                        [0, 1, 1]
  PASS  Explain merges both vocabularies   [('not good', -1.4800425111081226), ('at all', -1.1217545117696501), ('not', -0.2827909713509776)]
  PASS  Artefact saved                     10.3 MB
───────────────────────────────────────────────────────────────
  4/4 checks passed


In [6]:
# ── Cell 0 (bootstrap): everything Cells 6–10 need, from artefacts ────────
import sys, json, time
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import joblib

from sklearn.base import clone
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
sys.path.insert(0, str(PROJECT_ROOT / "app"))

# the shipped classes live in app/sentiment.py (written by Cell 5)
from sentiment import NBLogCountRatio, SoftVoteEnsemble

# v1 pipeline (also the config template Cell 9 clones)
model = joblib.load(DATA_DIR / "sentiment_model.joblib")

# training data for the adaptation experiment
train = pd.read_csv(DATA_DIR / "imdb_train_clean.csv")
X_tr, y_tr = train["text"], train["label"]

print(f"bootstrap ok — train {len(train):,} rows · "
      f"v2 artefact exists: {(DATA_DIR / 'sentiment_model_v2.joblib').exists()}")

bootstrap ok — train 24,902 rows · v2 artefact exists: True


In [7]:
# ── Cell 6: The Khadim suite — per-sentence verdicts, v1 vs v2 ────────────
import joblib

SUITE = [
    ("clear-positive", 1, "This app completely transformed how I organize my daily schedule! The interface is clean, super fast, and customer support replied within minutes."),
    ("clear-negative", 0, "Absolute waste of money. The software crashes every five minutes, deleted my saved progress, and customer service completely ignored my emails."),
    ("mild-positive", 1, "The package arrived a couple of days late, but the product itself works reasonably well and meets most of my basic expectations for the price."),
    ("mild-negative", 0, "It gets the job done, but the materials feel a bit cheap, the buttons are stiff, and I am not sure how long it will actually last with daily use."),
    ("mixed-lean-pos", 1, "The setup process was frustrating and the manual was unhelpful, but once I got it running, the performance exceeded all my expectations."),
    ("mixed-lean-neg", 0, "The restaurant had beautiful decor and attentive servers, but the food arrived stone cold and tasted completely bland for such an expensive place."),
    ("sarcasm-neg", 0, "Oh brilliant, another software update that moved all the buttons, broke half my plugins, and made the battery drain twice as fast. Outstanding work!"),
    ("sarcasm-pos", 1, "I was fully expecting this cheap laptop to break on day one, but somehow it hasn't given me a single issue in six months. Color me surprised."),
    ("double-negation", 1, "I cannot say that I am unhappy with this purchase. It is not cheap, but it definitely does not fail to deliver on its core promises."),
    ("pos-words-neg", 0, "They promised top-tier quality, fast shipping, and world-class reliability, but what I actually received was a cheap knockoff that broke instantly."),
]

v1 = joblib.load(DATA_DIR / "sentiment_model.joblib")
v2 = joblib.load(DATA_DIR / "sentiment_model_v2.joblib")

def verdict(m, text):
    p = m.predict_proba([text])[0][1]
    return ("POS" if p >= 0.5 else "NEG"), p

suite_score = {"v1": 0, "v2": 0}
suite_rows = []
print(f"{'category':<16}{'truth':<7}{'v1':<14}{'v2':<14}")
print("─" * 55)
for cat, truth, text in SUITE:
    row = {"cat": cat, "truth": truth}
    for name, m in [("v1", v1), ("v2", v2)]:
        lbl, p = verdict(m, text)
        ok = (lbl == "POS") == bool(truth)
        suite_score[name] += ok
        row[name] = f"{'✓' if ok else '✗'} {lbl} {p:.0%}"
    suite_rows.append(row)
    print(f"{cat:<16}{'POS' if truth else 'NEG':<7}{row['v1']:<14}{row['v2']:<14}")
print("─" * 55)
print(f"{'SUITE SCORE':<23}{suite_score['v1']}/10{'':<9}{suite_score['v2']}/10")

category        truth  v1            v2            
───────────────────────────────────────────────────────
clear-positive  POS    ✗ NEG 47%     ✗ NEG 46%     
clear-negative  NEG    ✓ NEG 3%      ✓ NEG 4%      
mild-positive   POS    ✓ POS 66%     ✓ POS 64%     
mild-negative   NEG    ✗ POS 58%     ✗ POS 59%     
mixed-lean-pos  POS    ✓ POS 79%     ✓ POS 70%     
mixed-lean-neg  NEG    ✗ POS 52%     ✗ POS 51%     
sarcasm-neg     NEG    ✗ POS 72%     ✗ POS 65%     
sarcasm-pos     POS    ✓ POS 65%     ✓ POS 58%     
double-negation POS    ✗ NEG 31%     ✗ NEG 44%     
pos-words-neg   NEG    ✓ NEG 44%     ✓ NEG 45%     
───────────────────────────────────────────────────────
SUITE SCORE            5/10         5/10


In [8]:
# ── Cell 7: UCI Sentiment Labelled Sentences — Amazon, Yelp, IMDB ─────────
import urllib.request, zipfile, io

UCI = ("https://archive.ics.uci.edu/ml/machine-learning-databases/00331/"
       "sentiment%20labelled%20sentences.zip")
zpath = DATA_DIR / "raw" / "uci_sentiment.zip"
zpath.parent.mkdir(parents=True, exist_ok=True)
if not zpath.exists():
    urllib.request.urlretrieve(UCI, zpath)

ood = {}
with zipfile.ZipFile(zpath) as z:
    for domain, fname in [("amazon", "amazon_cells_labelled.txt"),
                          ("yelp", "yelp_labelled.txt"),
                          ("imdb_sent", "imdb_labelled.txt")]:
        member = next(n for n in z.namelist() if n.endswith(fname))
        rows = []
        for line in io.TextIOWrapper(z.open(member), encoding="utf-8"):
            if "\t" in line:
                t, lbl = line.rsplit("\t", 1)
                rows.append((t.strip(), int(lbl)))
        ood[domain] = pd.DataFrame(rows, columns=["text", "label"])
        print(f"  {domain:<10} {len(ood[domain]):,} sentences  "
              f"(positive share {ood[domain]['label'].mean():.0%})")

  amazon     1,000 sentences  (positive share 50%)
  yelp       1,000 sentences  (positive share 50%)
  imdb_sent  1,000 sentences  (positive share 50%)


In [9]:
# ── Cell 8: Out-of-domain accuracy — the C3 number ────────────────────────
ood_acc = {}
print(f"{'domain':<12}{'v1 (tfidf-LR)':<16}{'v2 (ensemble)':<16}")
print("─" * 44)
for domain, frame in ood.items():
    accs = {}
    for name, m in [("v1", v1), ("v2", v2)]:
        acc = (m.predict(frame["text"]) == frame["label"]).mean()
        accs[name] = acc
    ood_acc[domain] = accs
    print(f"{domain:<12}{accs['v1']:<16.1%}{accs['v2']:<16.1%}")
print("─" * 44)
print("in-domain reference: v1 90.5% · v2 91.3% (full-review test)")

# the poison list: which movie-dialect features drive OOD errors?
from collections import Counter
poison = Counter()
for domain in ["amazon", "yelp"]:
    frame = ood[domain]
    preds = v2.predict(frame["text"])
    for text, lbl, pred in zip(frame["text"], frame["label"], preds):
        if pred != lbl:
            for feat, c in v2.explain(text, top=3):
                if (c > 0) == (pred == 1):          # features that pushed the wrong way
                    poison[feat] += 1
print("\nTop features driving OOD errors (the movie dialect):")
for feat, n in poison.most_common(12):
    print(f"  {feat:<18} {n}")

domain      v1 (tfidf-LR)   v2 (ensemble)   
────────────────────────────────────────────
amazon      78.0%           77.0%           
yelp        79.2%           78.3%           
imdb_sent   86.1%           86.5%           
────────────────────────────────────────────
in-domain reference: v1 90.5% · v2 91.3% (full-review test)

Top features driving OOD errors (the movie dialect):
  will               17
  very               16
  and it             12
  any                10
  still              10
  my                 10
  impressed          9
  it                 9
  bit                9
  also               9
  no                 8
  well               8


In [10]:
# ── Cell 9: Few-shot domain adaptation — 500 sentences per domain ─────────
# Protocol: UCI split in half — first 500 per domain may train, last 500 only
# evaluate. In-domain cost measured by CV, not by spending the IMDB test set.
from sklearn.base import clone
from sklearn.model_selection import cross_val_score

REP = 10   # tiny data gets weight by repetition (10 × 500 = 5,000 rows/domain)
aug_texts, aug_labels = [], []
ood_train, ood_eval = {}, {}
for domain in ["amazon", "yelp"]:
    half = len(ood[domain]) // 2
    ood_train[domain] = ood[domain].iloc[:half]
    ood_eval[domain]  = ood[domain].iloc[half:]
    aug_texts += list(ood_train[domain]["text"]) * REP
    aug_labels += list(ood_train[domain]["label"]) * REP

X_aug = pd.concat([X_tr, pd.Series(aug_texts)], ignore_index=True)
y_aug = pd.concat([y_tr, pd.Series(aug_labels)], ignore_index=True)
print(f"Augmented train: {len(X_aug):,} rows "
      f"({len(aug_texts):,} adapted, rep×{REP})")

m1a = clone(model).fit(X_aug, y_aug)
m2a = Pipeline([
    ("vec", CountVectorizer(ngram_range=(1, 2), min_df=5, binary=True)),
    ("nb",  NBLogCountRatio()),
    ("clf", LogisticRegression(C=0.5, max_iter=2000, solver="liblinear")),
]).fit(X_aug, y_aug)
ens_a = SoftVoteEnsemble([m1a, m2a])

print(f"\n{'eval set':<14}{'v2 (movie-only)':<18}{'v2 + adaptation':<18}")
print("─" * 50)
adapt_results = {}
for domain in ["amazon", "yelp"]:
    frame = ood_eval[domain]
    before = (v2.predict(frame["text"]) == frame["label"]).mean()
    after  = (ens_a.predict(frame["text"]) == frame["label"]).mean()
    adapt_results[domain] = (before, after)
    print(f"{domain+' (held)':<14}{before:<18.1%}{after:<18.1%}")

# in-domain cost check — CV on the movie training data only (no test spend)
cv_cost = cross_val_score(clone(model), X_aug, y_aug, cv=3,
                          scoring="accuracy", n_jobs=-1).mean()
print(f"\nAugmented-train CV (mixed): {cv_cost:.1%}  "
      "(watch for collapse vs ~90.4 movie-only CV — mild moves are fine)")

Augmented train: 34,902 rows (10,000 adapted, rep×10)

eval set      v2 (movie-only)   v2 + adaptation   
──────────────────────────────────────────────────
amazon (held) 73.4%             83.2%             
yelp (held)   78.0%             84.2%             

Augmented-train CV (mixed): 85.7%  (watch for collapse vs ~90.4 movie-only CV — mild moves are fine)


In [11]:
# ── Cell 10: C3 SUMMARY — paste this block back ───────────────────────────
print("═" * 58)
print("C3 RESULTS SUMMARY")
print("═" * 58)
print(f"Suite (10 sentences):    v1 {suite_score['v1']}/10   v2 {suite_score['v2']}/10")
print(f"Suite fails v2:          "
      f"{[r['cat'] for r in suite_rows if r['v2'].startswith('✗')]}")
print("-" * 58)
for domain in ["amazon", "yelp", "imdb_sent"]:
    a = ood_acc[domain]
    print(f"OOD {domain:<12} v1 {a['v1']:.1%}   v2 {a['v2']:.1%}")
print("-" * 58)
for domain, (before, after) in adapt_results.items():
    print(f"Adaptation {domain:<9} {before:.1%} → {after:.1%}   "
          f"(Δ {after-before:+.1%})")
print(f"Mixed-train CV check:    {cv_cost:.1%}")
print("-" * 58)
print("Top 5 poison features:  ", [f for f, _ in poison.most_common(5)])
print("═" * 58)

══════════════════════════════════════════════════════════
C3 RESULTS SUMMARY
══════════════════════════════════════════════════════════
Suite (10 sentences):    v1 5/10   v2 5/10
Suite fails v2:          ['clear-positive', 'mild-negative', 'mixed-lean-neg', 'sarcasm-neg', 'double-negation']
----------------------------------------------------------
OOD amazon       v1 78.0%   v2 77.0%
OOD yelp         v1 79.2%   v2 78.3%
OOD imdb_sent    v1 86.1%   v2 86.5%
----------------------------------------------------------
Adaptation amazon    73.4% → 83.2%   (Δ +9.8%)
Adaptation yelp      78.0% → 84.2%   (Δ +6.2%)
Mixed-train CV check:    85.7%
----------------------------------------------------------
Top 5 poison features:   ['will', 'very', 'and it', 'any', 'still']
══════════════════════════════════════════════════════════


In [12]:
# ── Cell 11: v3 = multi-domain ensemble — build from artefacts, ship ──────
import urllib.request, zipfile, io

# UCI halves (re-derived deterministically — same split as Cell 9)
UCI = ("https://archive.ics.uci.edu/ml/machine-learning-databases/00331/"
       "sentiment%20labelled%20sentences.zip")
zpath = DATA_DIR / "raw" / "uci_sentiment.zip"
zpath.parent.mkdir(parents=True, exist_ok=True)
if not zpath.exists():
    urllib.request.urlretrieve(UCI, zpath)
ood = {}
with zipfile.ZipFile(zpath) as z:
    for domain, fname in [("amazon", "amazon_cells_labelled.txt"),
                          ("yelp", "yelp_labelled.txt"),
                          ("imdb_sent", "imdb_labelled.txt")]:
        member = next(n for n in z.namelist() if n.endswith(fname))
        rows = [(t.strip(), int(l)) for t, l in
                (ln.rsplit("\t", 1) for ln in
                 io.TextIOWrapper(z.open(member), encoding="utf-8") if "\t" in ln)]
        ood[domain] = pd.DataFrame(rows, columns=["text", "label"])

REP = 10
aug_texts, aug_labels, ood_eval = [], [], {}
for domain in ["amazon", "yelp"]:
    half = len(ood[domain]) // 2
    aug_texts += list(ood[domain].iloc[:half]["text"]) * REP
    aug_labels += list(ood[domain].iloc[:half]["label"]) * REP
    ood_eval[domain] = ood[domain].iloc[half:]

X_aug = pd.concat([X_tr, pd.Series(aug_texts)], ignore_index=True)
y_aug = pd.concat([y_tr, pd.Series(aug_labels)], ignore_index=True)

m1a = clone(model).fit(X_aug, y_aug)
m2a = Pipeline([
    ("vec", CountVectorizer(ngram_range=(1, 2), min_df=5, binary=True)),
    ("nb",  NBLogCountRatio()),
    ("clf", LogisticRegression(C=0.5, max_iter=2000, solver="liblinear")),
]).fit(X_aug, y_aug)
v3 = SoftVoteEnsemble([m1a, m2a])

joblib.dump(v3, DATA_DIR / "sentiment_model_v3.joblib")
print(f"v3 saved "
      f"({(DATA_DIR / 'sentiment_model_v3.joblib').stat().st_size/1e6:.1f} MB) — "
      f"trained on {len(X_aug):,} rows (movies + amazon/yelp first halves)")

v3 saved (10.5 MB) — trained on 34,902 rows (movies + amazon/yelp first halves)


In [13]:
# ── Cell 12: v3 measured — movie test (single shot), suite, OOD halves ────
df_all = pd.read_csv(DATA_DIR / "imdb_reviews.csv")
test = df_all[df_all["split"] == "test"].reset_index(drop=True)

movie_test = (v3.predict(test["text"]) == test["label"]).mean()

suite_v3 = 0
suite_fail_v3 = []
for cat, truth, text in SUITE:
    p = v3.predict_proba([text])[0][1]
    ok = (p >= 0.5) == bool(truth)
    suite_v3 += ok
    if not ok:
        suite_fail_v3.append(cat)

ood_v3 = {}
for domain, frame in ood_eval.items():
    ood_v3[domain] = (v3.predict(frame["text"]) == frame["label"]).mean()

In [14]:
# ── Cell 13: V3 SHIPPING SUMMARY — paste this block back ──────────────────
print("═" * 58)
print("V3 SHIPPING SUMMARY")
print("═" * 58)
print(f"Movie test (single shot):  {movie_test:.1%}   (v2: 91.3% · v1: 90.5%)")
print(f"Suite:                     {suite_v3}/10   (v1 5 · v2 5)")
print(f"Suite still failing:       {suite_fail_v3}")
for d in ["amazon", "yelp"]:
    print(f"OOD {d:<10} (held):     {ood_v3[d]:.1%}   (v2 was "
          f"{'73.4%' if d == 'amazon' else '78.0%'})")
print("═" * 58)

══════════════════════════════════════════════════════════
V3 SHIPPING SUMMARY
══════════════════════════════════════════════════════════
Movie test (single shot):  90.8%   (v2: 91.3% · v1: 90.5%)
Suite:                     5/10   (v1 5 · v2 5)
Suite still failing:       ['mild-negative', 'mixed-lean-neg', 'sarcasm-pos', 'double-negation', 'pos-words-neg']
OOD amazon     (held):     83.2%   (v2 was 73.4%)
OOD yelp       (held):     84.2%   (v2 was 78.0%)
══════════════════════════════════════════════════════════


In [15]:
# ── Cell 14: C4 setup — device check ──────────────────────────────────────
import torch
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification

DEVICE = ("mps" if torch.backends.mps.is_available()
          else "cuda" if torch.cuda.is_available() else "cpu")
print(f"torch {torch.__version__} · device: {DEVICE}")
if DEVICE == "cpu":
    print("⚠ CPU only — training will take ~40-60 min/epoch. "
          "Tell me before proceeding; we'll shrink the config.")

MODEL_NAME = "distilbert-base-uncased"
tok = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)
net = DistilBertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
net = net.to(DEVICE)
print(f"model loaded: {sum(p.numel() for p in net.parameters())/1e6:.0f}M parameters")

/opt/anaconda3/envs/u2pr311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch 2.13.0 · device: mps


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 21529.12it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


model loaded: 67M parameters


In [16]:
# ── Cell 15: Multi-domain training data → tensors ─────────────────────────
from torch.utils.data import DataLoader, TensorDataset

MAX_LEN, BATCH = 256, 16

# same mix as v3, but no repetition — transformers don't need the ×10 trick
texts = list(X_tr) + aug_texts[::10]          # every 10th = the originals once
labels = list(y_tr) + aug_labels[::10]
print(f"training rows: {len(texts):,} (movies {len(X_tr):,} + adapted 1,000)")

def encode(texts, labels=None):
    enc = tok(list(texts), truncation=True, padding="max_length",
              max_length=MAX_LEN, return_tensors="pt")
    if labels is None:
        return TensorDataset(enc["input_ids"], enc["attention_mask"])
    return TensorDataset(enc["input_ids"], enc["attention_mask"],
                         torch.tensor(list(labels)))

print("tokenizing… (~1 min)")
train_ds = encode(texts, labels)
train_dl = DataLoader(train_ds, batch_size=BATCH, shuffle=True)
print(f"{len(train_dl):,} steps per epoch")

training rows: 25,902 (movies 24,902 + adapted 1,000)
tokenizing… (~1 min)
1,619 steps per epoch


In [17]:
# ── Cell 16: Fine-tune, 1 epoch ───────────────────────────────────────────
import time

opt = torch.optim.AdamW(net.parameters(), lr=2e-5)
net.train()
t0, running = time.time(), 0.0
for step, (ids, mask, y) in enumerate(train_dl, 1):
    ids, mask, y = ids.to(DEVICE), mask.to(DEVICE), y.to(DEVICE)
    opt.zero_grad()
    out = net(input_ids=ids, attention_mask=mask, labels=y)
    out.loss.backward()
    opt.step()
    running += out.loss.item()
    if step % 100 == 0:
        print(f"  step {step:>5}/{len(train_dl)}  loss {running/100:.4f}  "
              f"({time.time()-t0:.0f}s)")
        running = 0.0
print(f"epoch done in {(time.time()-t0)/60:.1f} min")

out_dir = DATA_DIR / "distilbert_sentiment"
net.save_pretrained(out_dir); tok.save_pretrained(out_dir)
print(f"saved to {out_dir.name}/")

  step   100/1619  loss 0.5094  (21s)
  step   200/1619  loss 0.3091  (40s)
  step   300/1619  loss 0.2923  (59s)
  step   400/1619  loss 0.2917  (79s)
  step   500/1619  loss 0.2941  (99s)
  step   600/1619  loss 0.2512  (118s)
  step   700/1619  loss 0.2737  (141s)
  step   800/1619  loss 0.2666  (172s)
  step   900/1619  loss 0.2619  (222s)
  step  1000/1619  loss 0.2508  (262s)
  step  1100/1619  loss 0.2523  (295s)
  step  1200/1619  loss 0.2179  (326s)
  step  1300/1619  loss 0.2338  (356s)
  step  1400/1619  loss 0.2244  (386s)
  step  1500/1619  loss 0.2259  (417s)
  step  1600/1619  loss 0.2497  (447s)
epoch done in 7.5 min


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  8.76it/s]

saved to distilbert_sentiment/


In [18]:
# ── Cell 17: DistilBERT measured — test shot, suite, OOD halves ───────────
@torch.no_grad()
def bert_proba(texts, batch=64):
    net.eval()
    ps = []
    for i in range(0, len(texts), batch):
        enc = tok(list(texts[i:i+batch]), truncation=True, padding=True,
                  max_length=MAX_LEN, return_tensors="pt").to(DEVICE)
        logits = net(**enc).logits
        ps.append(torch.softmax(logits, dim=-1)[:, 1].cpu())
    return torch.cat(ps).numpy()

print("scoring the 25k movie test… (~2-5 min)")
t0 = time.time()
p_test = bert_proba(list(test["text"]))
bert_movie = ((p_test >= 0.5).astype(int) == test["label"]).mean()
print(f"movie test: {bert_movie:.1%}   ({(time.time()-t0)/60:.1f} min)")

bert_suite, bert_suite_fail = 0, []
for cat, truth, text in SUITE:
    ok = (bert_proba([text])[0] >= 0.5) == bool(truth)
    bert_suite += ok
    if not ok:
        bert_suite_fail.append(cat)

bert_ood = {}
for domain, frame in ood_eval.items():
    p = bert_proba(list(frame["text"]))
    bert_ood[domain] = ((p >= 0.5).astype(int) == frame["label"]).mean()

scoring the 25k movie test… (~2-5 min)
movie test: 91.3%   (2.0 min)


In [19]:
# ── Cell 18: C4 SUMMARY — paste this block back ───────────────────────────
print("═" * 60)
print("C4 (DistilBERT, 1 epoch) SUMMARY")
print("═" * 60)
print(f"Movie test:   {bert_movie:.1%}   (v3 90.8 · v2 91.3 · published ~92.8)")
print(f"Suite:        {bert_suite}/10   (classical best: 5/10)")
print(f"Suite fails:  {bert_suite_fail}")
for d in ["amazon", "yelp"]:
    print(f"OOD {d:<8} {bert_ood[d]:.1%}   (v3: {'83.2%' if d=='amazon' else '84.2%'})")
print("═" * 60)

════════════════════════════════════════════════════════════
C4 (DistilBERT, 1 epoch) SUMMARY
════════════════════════════════════════════════════════════
Movie test:   91.3%   (v3 90.8 · v2 91.3 · published ~92.8)
Suite:        7/10   (classical best: 5/10)
Suite fails:  ['mixed-lean-pos', 'sarcasm-neg', 'sarcasm-pos']
OOD amazon   92.2%   (v3: 83.2%)
OOD yelp     92.0%   (v3: 84.2%)
════════════════════════════════════════════════════════════
